#### By: Peyman Shahidi
#### Created: Aug 1, 2025

<br>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

main_folder_path = ".."
output_plot_path = f"{main_folder_path}/writeup/plots"
os.makedirs(output_plot_path, exist_ok=True)

# Short-run chaining example under the COST objective. Every step has unit skill,
# so a task's cost equals its (expected) time. Step 1 is short but hard for AI;
# step 2 is moderately time-consuming but easy for AI. The firm minimizes total cost.
# Numbers chosen so the marginal benefit at the SECOND (chaining) threshold exceeds
# the first (augmentation) threshold: chaining brings an extra step under AI, so the
# cost falls more steeply (alpha^-7 vs alpha^-1).
t1m, t2m = 1, 2   # manual costs (= time, since skill = 1)
t1a, t2a = 1, 1   # AI management cost per attempt
d1, d2 = 6, 1     # AI difficulty: q_i = alpha**d_i

alpha = np.linspace(0.001, 1, 4000)

both_manual = (t1m + t2m) * np.ones_like(alpha)
s1m_s2aug   = t1m + t2a / alpha**d2          # 1 + 1/alpha
s1aug_s2m   = t1a / alpha**d1 + t2m
both_aug    = t1a / alpha**d1 + t2a / alpha**d2
chain       = t2a / alpha**(d1 + d2)         # alpha**-7 : steps 1-2 as one AI chain

costs = np.vstack([both_manual, s1m_s2aug, s1aug_s2m, both_aug, chain])
min_cost = costs.min(axis=0)

bp1 = alpha[np.argmin(np.abs(both_manual - s1m_s2aug))]   # ~0.50
bp2 = alpha[np.argmin(np.abs(s1m_s2aug - chain))]         # ~0.90

# Shared styling so the two panels look identical.
FIGSIZE   = (8, 6)
LABEL_FS  = 15   # axis labels
TICK_FS   = 12   # tick labels
LEGEND_FS = 11   # legend entries
THRESH_FS = 10   # alpha-threshold annotations

# ---------- Panel (a): total cost by AI strategy ----------
plt.figure(figsize=FIGSIZE)
plt.plot(alpha, both_manual, label="Both steps manual", alpha=0.55, color="C0")
plt.plot(alpha, s1m_s2aug, label="Step 1 manual, Step 2 augmented", alpha=0.55, color="C1")
plt.plot(alpha, both_aug, label="Both steps augmented", alpha=0.55, color="C3")
plt.plot(alpha, chain, label="Steps 1--2 chained (1 automated, 2 augmented)", alpha=0.55, color="C4")
plt.plot(alpha, min_cost, label="Minimum cost (optimal AI strategy)", color="black", linewidth=4)
for bp in (bp1, bp2):
    plt.axvline(bp, color="gray", linestyle="--", linewidth=1, alpha=0.8)
    plt.text(bp, 0.2, rf"$\alpha={bp:.2f}$", ha="center", va="bottom", fontsize=THRESH_FS, backgroundcolor="white")
plt.xlabel(r"AI Quality Level $\alpha$", fontsize=LABEL_FS)
plt.ylabel("Total Cost", fontsize=LABEL_FS)
plt.ylim(-0.5, 10)
plt.xlim(0.1, 1)
plt.tick_params(labelsize=TICK_FS)
plt.legend(loc="upper left", fontsize=LEGEND_FS)
plt.tight_layout()
plt.savefig(f"{output_plot_path}/example5_costs.png", dpi=300)
plt.show()

# ---------- Panel (b): marginal benefit of improving alpha ----------
mb_aug = t2a * d2 / alpha**(d2 + 1)               # 1/alpha^2
mb_chain = t2a * (d1 + d2) / alpha**(d1 + d2 + 1)  # 7/alpha^8
mb_opt = np.piecewise(
    alpha,
    [alpha <= bp1, (alpha > bp1) & (alpha <= bp2), alpha > bp2],
    [0, lambda a: t2a * d2 / a**(d2 + 1), lambda a: t2a * (d1 + d2) / a**(d1 + d2 + 1)],
)
plt.figure(figsize=FIGSIZE)
plt.plot(alpha, np.zeros_like(alpha), label="Both steps manual", alpha=0.55, color="C0")
plt.plot(alpha, mb_aug, label="Step 1 manual, Step 2 augmented", alpha=0.55, color="C1")
plt.plot(alpha, mb_chain, label="Steps 1--2 chained", alpha=0.55, color="C4")
plt.plot(alpha, mb_opt, label="Optimal AI strategy", color="black", linewidth=4)
for bp in (bp1, bp2):
    plt.axvline(bp, color="gray", linestyle="--", linewidth=1, alpha=0.8)
    plt.text(bp, 7.5, rf"$\alpha={bp:.2f}$", ha="center", va="center", fontsize=THRESH_FS, backgroundcolor="white")
plt.xlabel(r"AI Quality Level $\alpha$", fontsize=LABEL_FS)
plt.ylabel(r"Marginal Benefit of Increasing $\alpha$", fontsize=LABEL_FS)
plt.ylim(-0.5, 18)
plt.xlim(0.1, 1)
plt.tick_params(labelsize=TICK_FS)
plt.legend(loc="upper left", fontsize=LEGEND_FS)
plt.tight_layout()
plt.savefig(f"{output_plot_path}/example5_marginalBenefit.png", dpi=300)
plt.show()
